In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_PATH = Path().resolve().parent.parent / "data" / "raw" / "finanças"
OUTPUT_DATA_PATH = Path().resolve().parent.parent / "data" / "processed"

# Despesas
def processar_despesas() -> pd.DataFrame:
    despesas_2018 = pd.read_csv(
        RAW_DATA_PATH / 'despesas_contratadas_candidatos_2018_BRASIL.csv',
        sep=';',
        encoding='latin1',
    )
    despesas_2022 = pd.read_csv(
        RAW_DATA_PATH / 'despesas_contratadas_candidatos_2022_BRASIL.csv',
        sep=';',
        encoding='latin1',
    )
    def tratar_despesas(df_despesas):
        df_despesas.columns = df_despesas.columns.str.lower()
        df_despesas['vr_despesa_contratada'] = (
            df_despesas['vr_despesa_contratada']
            .str.replace(',', '.')
            .astype(float)
        )
        for col in ['ds_cargo', 'ds_cargo_fornecedor']:
            df_despesas[col] = df_despesas[col].str.upper()
        df_despesas = (
            df_despesas
            .groupby([
                'ano_eleicao', 'sg_uf', 'nr_candidato',
                'nm_candidato', 'sg_partido', 'ds_cargo',
                'sg_uf_fornecedor', 'nr_candidato_fornecedor', 
                'nm_fornecedor', 'sg_partido_fornecedor',
                'ds_cargo_fornecedor', 'ds_origem_despesa', 'dt_despesa'
            ], as_index=False)
            .agg({'vr_despesa_contratada':'sum'})
            .sort_values('vr_despesa_contratada', ascending=False)
        )
        return df_despesas
    despesas_2018_resumo = tratar_despesas(despesas_2018)
    despesas_2022_resumo = tratar_despesas(despesas_2022)
    despesas = pd.concat([despesas_2018_resumo, despesas_2022_resumo], axis=0)
    despesas['dt_despesa'] = pd.to_datetime(despesas['dt_despesa'], format='%d/%m/%Y')
    despesas.to_parquet(OUTPUT_DATA_PATH / "despesas.parquet", index=False)
    return despesas


processar_despesas()

,ano_eleicao,sg_uf,nr_candidato,nm_candidato,sg_partido,ds_cargo,sg_uf_fornecedor,nr_candidato_fornecedor,nm_fornecedor,sg_partido_fornecedor,ds_cargo_fornecedor,ds_origem_despesa,dt_despesa,vr_despesa_contratada
98001,2018,BR,15,HENRIQUE DE CAMPOS MEIRELLES,MDB,PRESIDENTE,#NULO#,-1,D18 PRODUCAO DE FILMES SPE LTDA,#NULO,#NULO,"Produção de programas de rádio, televisão ou v...",2018-08-16,15080000.00
98745,2018,BR,15,HENRIQUE DE CAMPOS MEIRELLES,MDB,PRESIDENTE,#NULO#,-1,NACIONAL COMUNICACAO SPE LTDA,#NULO,#NULO,"Produção de programas de rádio, televisão ou v...",2018-08-23,13490000.00
314738,2018,MG,45,ANTONIO AUGUSTO JUNHO ANASTASIA,PSDB,GOVERNADOR,#NULO#,-1,2018 COMUNICAÇÃO SPE LTDA,#NULO,#NULO,"Produção de programas de rádio, televisão ou v...",2018-08-24,7294000.00
97625,2018,BR,13,LUIZ INACIO LULA DA SILVA,PT,PRESIDENTE,#NULO#,-1,M. ROMANO COMUNICAÇÃO LTDA ME,#NULO,#NULO,"Produção de programas de rádio, televisão ou v...",2018-08-27,5236000.00
97698,2018,BR,13,LUIZ INACIO LULA DA SILVA,PT,PRESIDENTE,#NULO#,-1,RENTAL LOCAÇÃO DE BENS MÓVEIS LTDA,#NULO,#NULO,"Produção de programas de rádio, televisão ou v...",2018-08-17,4900000.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933272,2022,PR,27444,FABIO FOCKINK,DC,DEPUTADO ESTADUAL,#NULO#,-1,GRAFICA ESCALA LTDA,#NULO,#NULO,Diversas a especificar,2022-09-09,0.01
619858,2022,MG,55123,WILSON ROBERTO BATISTA,PSD,DEPUTADO ESTADUAL,#NULO#,-1,MW TRANSPORTES LTDA,#NULO,#NULO,Correspondências e despesas postais,2022-08-23,0.01
1397066,2022,SP,2227,ROSANA DE OLIVEIRA VALLE,PL,DEPUTADO FEDERAL,#NULO#,-1,#NULO,#NULO,#NULO,"Impostos, contribuições e taxas",2022-09-29,0.01
933248,2022,PR,27270,MARCOS ROGERIO DOMBROWSKI,DC,DEPUTADO ESTADUAL,#NULO#,-1,IZADORA H REIS,#NULO,#NULO,Cessão ou locação de veículos,2022-09-16,0.01
